# Drone Forensic → MISP Event

This notebook walks through the pipeline **one artifact at a time**, showing for each:

1. what the extractor returns (the in-memory dict)
2. which MISP object template we use
3. how each extracted field maps to an object attribute (`object_relation → value`)

All encoder functions are imported from `misp/drone_to_misp.py` — the CLI uses the exact same functions, so running this notebook produces an equivalent MISP Event.

Switch `CASE` below to run against `forensic-01` or `forensic-02`.

In [ ]:
import inspect
import sys
from pathlib import Path

from IPython.display import Code

REPO = Path().resolve().parent
sys.path.insert(0, str(REPO / "misp"))

import drone_to_misp as mdf
from pymisp import MISPEvent

def show_src(fn):
    """Render the source of an encoder function with Python syntax highlighting."""
    return Code(inspect.getsource(fn), language='python')

def show_attrs(obj):
    """Tabulate a MISP object's attributes — what the encoder just produced."""
    print(f"<MISPObject name={obj.name!r} uuid={obj.uuid[:8]} attrs={len(obj.attributes)} refs={len(obj.references)}>")
    for a in obj.attributes:
        v = a.value
        if isinstance(v, str) and len(v) > 60:
            v = v[:57] + "..."
        print(f"  {a.object_relation:<22} = {v}")
    for r in obj.references:
        print(f"  --[{r.relationship_type}]--> {r.referenced_uuid[:8]}")

In [24]:
CASE = "forensic-02"

case_dir = REPO / "workshop" / CASE
fc_path   = next(case_dir.glob("fc_*.bin"), None)
bb_path   = next(case_dir.glob("blackbox_*.bin"), None)
elrs_path = next(case_dir.glob("rx_*.bin"), None)

print("FC      :", fc_path and fc_path.name)
print("Blackbox:", bb_path and bb_path.name)
print("ELRS    :", elrs_path and elrs_path.name)

FC      : fc_9e5acfc3258f1ed9537230d7bc87a915.bin
Blackbox: blackbox_3ede1ae6dc783a62a4ea9dcac859aa58.bin
ELRS    : rx_affdc5b9135e961b7aa5fb252938f49c.bin


## 1. Extract — intermediate dict

`run_extraction` wraps the three existing extractors (`fc/src/`, `blackbox/`, `elrs/src/`) and returns an in-memory dict keyed by artifact type. Missing artifacts are simply absent. **Nothing is cached to disk** — this dict is the direct input to the encoder.

In [25]:
extracted = mdf.run_extraction(fc_path, bb_path, elrs_path)
print("Top-level keys:", list(extracted.keys()))

[FC] extracting fc_9e5acfc3258f1ed9537230d7bc87a915.bin ...
[blackbox] extracting blackbox_3ede1ae6dc783a62a4ea9dcac859aa58.bin ...
[ELRS] extracting rx_affdc5b9135e961b7aa5fb252938f49c.bin ...


Top-level keys: ['fc', 'blackbox', 'elrs']


## 2. Create an empty Event

One case → one `MISPEvent`. We populate it by calling the encoder's per-artifact builders and inspecting each resulting object. `build_event` runs this same sequence end-to-end — we reproduce it step by step for pedagogy.

In [26]:
event = MISPEvent()
event.info = f"Drone forensic — {CASE}"
print(event.info)

Drone forensic — forensic-02


## 3. Flight controller dump

The FC pipeline is the richest. From a single ROM dump we derive up to five MISP objects:

| Object         | Describes                                            |
|----------------|------------------------------------------------------|
| `file`         | the raw `.bin` dump itself (base64 attachment)       |
| `iot-device`   | the flight controller **board** (STM32 / target / firmware family) |
| `iot-firmware` | the firmware **artifact** (hash, version)            |
| `uav`          | the **drone** as a whole (anchor of the graph)       |
| `gpx`          | INAV mission waypoints, rendered in-memory (INAV only) |

### 3.1 What the FC extractor returns

YARA rules in `fc/yara/` fingerprint the firmware family, then the board/chipset. The FLASH_CONFIG region (STM32 memory map entry keyed by the chipset) is sliced out of the dump and decoded to recover the craft name, pilot name, and (on INAV) the mission waypoints.

In [27]:
fc = extracted.get("fc", {})
print("file     :", {k: fc['file'][k] for k in ('filename','md5','size')})
print("firmware :", fc.get("firmware"))
print("craft    :", fc.get("craft_name"))
print("pilot    :", fc.get("pilot_name"))
print("waypoints:", len(fc.get("waypoints", [])), "(first 2 shown)")
for wp in fc.get("waypoints", [])[:2]:
    print("   ", wp)

file     : {'filename': 'fc_9e5acfc3258f1ed9537230d7bc87a915.bin', 'md5': '9e5acfc3258f1ed9537230d7bc87a915', 'size': 524288}
firmware : {'family': 'BETAFLIGHT', 'target': 'AXISFLYINGF7', 'chipset': 'STM32F722', 'build_macro': ''}
craft    : MK1
pilot    : Thanat0s
waypoints: 0 (first 2 shown)


### 3.2 `file` — the FC dump itself

The whole raw dump is base64-embedded in the `attachment` field. The event is self-contained: no external file dependency.

**Mapping**

| `fc.file[...]`   | `file.<object_relation>` |
|------------------|--------------------------|
| `filename`       | `filename`               |
| `md5`, `sha256`  | `md5`, `sha256`          |
| `size`           | `size-in-bytes`          |
| raw bytes (b64)  | `attachment`             |

In [ ]:
show_src(mdf._build_file_object)

In [28]:
fc_file = mdf._build_file_object(fc["file"])
event.add_object(fc_file)
show_attrs(fc_file)

<MISPObject name='file' uuid=15627a15 attrs=5 refs=0>
  filename               = fc_9e5acfc3258f1ed9537230d7bc87a915.bin
  md5                    = 9e5acfc3258f1ed9537230d7bc87a915
  sha256                 = 693c0d3c3e566260f3f8b2390c736d092f2b245a56ce6c8136a6414f2...
  size-in-bytes          = 524288
  attachment             = fc_9e5acfc3258f1ed9537230d7bc87a915.bin


### 3.3 `iot-device` — the board

| `fc.firmware[...]` | `iot-device.<object_relation>` |
|--------------------|--------------------------------|
| `target`           | `model` (e.g. `AETH743Basic`)   |
| `chipset`          | `architecture` (e.g. `STM32H743`) |
| `family`           | `platform` (INAV / BETAFLIGHT / ARDUPILOT) |

In [ ]:
show_src(mdf._build_iot_device)

In [29]:
iot_device = mdf._build_iot_device(fc)
if iot_device is not None:
    event.add_object(iot_device)
    show_attrs(iot_device)

<MISPObject name='iot-device' uuid=3cc3d8eb attrs=3 refs=0>
  model                  = AXISFLYINGF7
  architecture           = STM32F722
  platform               = BETAFLIGHT


### 3.4 `iot-firmware` — the firmware artifact

Describes the firmware blob itself (hashes, `build_macro` as `version`, `format` = `raw`). `iot-device` describes the hardware that runs it.

In [ ]:
show_src(mdf._build_iot_firmware)

In [30]:
iot_firmware = mdf._build_iot_firmware(fc)
if iot_firmware is not None:
    event.add_object(iot_firmware)
    show_attrs(iot_firmware)

<MISPObject name='iot-firmware' uuid=0892ea17 attrs=5 refs=0>
  filename               = fc_9e5acfc3258f1ed9537230d7bc87a915.bin
  md5                    = 9e5acfc3258f1ed9537230d7bc87a915
  sha256                 = 693c0d3c3e566260f3f8b2390c736d092f2b245a56ce6c8136a6414f2...
  size-in-bytes          = 524288
  format                 = raw


### 3.5 `uav` — the drone (anchor)

Every other object references `uav`. When both FC and blackbox are present, **FC is authoritative** — blackbox only fills fields FC didn't produce (typically `log_start_datetime` / `firmware_version`). `_merge_craft_info` applies that dedup rule.

The `uav` template requires one of `model` / `manufacturer` / `serial-number`. We populate `model` from the FC target (falling back to the craft name), and keep the user-chosen craft name in `craftname`.

In [ ]:
show_src(mdf._merge_craft_info)

In [ ]:
show_src(mdf._build_uav_object)

In [31]:
craft_info = mdf._merge_craft_info(fc, extracted.get("blackbox"))
print("merged craft_info:", craft_info)
print()
uav = mdf._build_uav_object(craft_info, fc)
event.add_object(uav)
show_attrs(uav)

merged craft_info: {'craft_name': 'MK1', 'pilot_name': 'Thanat0s', 'firmware_version': 'Betaflight 2025.12.1 (85d201376) STM32F7X2'}

<MISPObject name='uav' uuid=f90ad24f attrs=5 refs=0>
  model                  = AXISFLYINGF7
  craftname              = MK1
  operator               = Thanat0s
  firmware-version       = Betaflight 2025.12.1 (85d201376) STM32F7X2
  variant                = BETAFLIGHT


### 3.6 `gpx` — waypoints as an in-memory GPX trace

INAV mission waypoints are rendered to GPX **in-memory** (not from a `.gpx` file on disk) and embedded as an attachment. If the FC has no waypoints (Betaflight, or empty INAV mission), this object is skipped and we fall back to the blackbox trace in §4.

In [ ]:
show_src(mdf._build_gpx_object)

In [32]:
import hashlib

fc_gpx = None
if fc.get("gpx_text"):
    gpx_bytes = fc["gpx_text"].encode("utf-8")
    fc_gpx = mdf._build_gpx_object(
        fc["gpx_text"],
        filename=f"{Path(fc['file']['filename']).stem}_waypoints.gpx",
        hashes={
            "md5":    hashlib.md5(gpx_bytes).hexdigest(),
            "sha256": hashlib.sha256(gpx_bytes).hexdigest(),
            "size":   len(gpx_bytes),
        },
        waypoint_count=len(fc.get("waypoints", [])),
    )
    event.add_object(fc_gpx)
    show_attrs(fc_gpx)
else:
    print("No FC GPX for this case — will try blackbox next.")

No FC GPX for this case — will try blackbox next.


## 4. Blackbox log

Produces a `file` (the dump), optionally a `gpx` (if FC didn't already provide one), and up to two `geolocation` objects for arming / disarming positions.

In [33]:
bb = extracted.get("blackbox", {})
print("file :", {k: bb['file'][k] for k in ('filename','md5','size')})
for i, log in enumerate(bb.get("logs", [])[:3]):
    print(f"log #{i+1}: fw={log.get('firmware_revision')!r} craft={log.get('craft_name')!r}"
          f" start={log.get('log_start_datetime')!r} gps={log.get('has_gps')}")
    for k in ("arming_coord", "disarmed_coord"):
        if log.get(k):
            print(f"   {k}: {log[k]}")

file : {'filename': 'blackbox_3ede1ae6dc783a62a4ea9dcac859aa58.bin', 'md5': '3ede1ae6dc783a62a4ea9dcac859aa58', 'size': 16777216}
log #1: fw='Betaflight 2025.12.1 (85d201376) STM32F7X2' craft='MK1' start='2026-04-06T16:09:51.073+00:00' gps=True
   arming_coord: {'lat': '49.4464139', 'lon': '5.9128042'}
   disarmed_coord: {'lat': '49.4440938', 'lon': '5.9107732'}


In [34]:
blackbox_file = mdf._build_file_object(bb["file"])
event.add_object(blackbox_file)
show_attrs(blackbox_file)

<MISPObject name='file' uuid=dd907b1a attrs=5 refs=0>
  filename               = blackbox_3ede1ae6dc783a62a4ea9dcac859aa58.bin
  md5                    = 3ede1ae6dc783a62a4ea9dcac859aa58
  sha256                 = 9963ae588b5969fb1a368f6ee5efb2e033eb1eda858d62e3aa4aa77a5...
  size-in-bytes          = 16777216
  attachment             = blackbox_3ede1ae6dc783a62a4ea9dcac859aa58.bin


### 4.1 Fallback `gpx` from the blackbox flight path

Built only if the FC didn't already produce one. This is the typical Betaflight case (no mission waypoints, but sampled GPS during flight).

In [35]:
blackbox_gpx = None
if bb.get("gpx_text") and fc_gpx is None:
    blackbox_gpx = mdf._build_gpx_object(
        bb["gpx_text"],
        filename=bb.get("gpx_filename", "flight.gpx"),
        hashes=bb["gpx_hashes"],
        waypoint_count=0,
    )
    event.add_object(blackbox_gpx)
    show_attrs(blackbox_gpx)
else:
    print("No blackbox GPX attached (FC already provided one, or no GPS).")

<MISPObject name='gpx' uuid=e7ad83ad attrs=13 refs=0>
  filename               = blackbox_3ede1ae6dc783a62a4ea9dcac859aa58.bin.01.gps.gpx
  md5                    = 94fe1f93cfc693d9344383fec139aa68
  sha256                 = 0d995650509f44d131e539c94bdfdd697af0ba316d782c97b40fa3f21...
  size-in-bytes          = 126674
  gpx-version            = 1.1
  mime-type              = application/gpx+xml
  attachment             = blackbox_3ede1ae6dc783a62a4ea9dcac859aa58.bin.01.gps.gpx
  creator                = misp_drone_forensic
  min-latitude           = 49.4440938
  max-latitude           = 49.4464228
  min-longitude          = 5.910443
  max-longitude          = 5.91482
  point-count            = 1180


### 4.2 `geolocation` — arming / disarming points

One object per fix. The `text` attribute distinguishes `arming position` from `disarming position`.

In [ ]:
show_src(mdf._build_geolocations)

In [36]:
geolocation_objs = mdf._build_geolocations(bb)
print(f"{len(geolocation_objs)} geolocation object(s)\n")
for _, geo in geolocation_objs:
    event.add_object(geo)
    show_attrs(geo)
    print()

2 geolocation object(s)

<MISPObject name='geolocation' uuid=a3aa2453 attrs=3 refs=0>
  latitude               = 49.4464139
  longitude              = 5.9128042
  text                   = arming position

<MISPObject name='geolocation' uuid=e66bd288 attrs=3 refs=0>
  latitude               = 49.4440938
  longitude              = 5.9107732
  text                   = disarming position



## 5. ELRS receiver config

| Extracted                  | Object              | Attribute                               |
|----------------------------|---------------------|-----------------------------------------|
| raw bytes, hashes          | `file`              | `filename`, `md5`, `sha256`, `attachment` |
| `wifi_ssid`, `wifi_password` | `wifi-connection`   | `ssid`, `password`                        |
| `uid` (6 bytes)            | `remote-controller` | `model`=`ExpressLRS receiver`, `serial-number`, `remote-controller-id` |

`wifi-connection` is only built if the config actually contained SSID/password — newer ELRS firmwares may not expose them in the dump.

In [37]:
elrs = extracted.get("elrs", {})
for e in elrs.get("entries", []):
    uid = e.get("uid")
    if not any((uid, e.get("wifi_ssid"), e.get("wifi_password"))):
        continue
    uid_str = "-".join(f"{b:02X}" for b in uid) if uid else None
    print(f"uid={uid_str}  ssid={e.get('wifi_ssid')!r}  password={e.get('wifi_password')!r}")

uid=55-49-9B-D7-0C-36  ssid=None  password=None


In [38]:
elrs_file = mdf._build_file_object(elrs["file"])
event.add_object(elrs_file)
show_attrs(elrs_file)

<MISPObject name='file' uuid=031dbbed attrs=5 refs=0>
  filename               = rx_affdc5b9135e961b7aa5fb252938f49c.bin
  md5                    = affdc5b9135e961b7aa5fb252938f49c
  sha256                 = 330d095b0c117f39620e0c3a3bb019af22804ee07afdf6649818e850c...
  size-in-bytes          = 2097152
  attachment             = rx_affdc5b9135e961b7aa5fb252938f49c.bin


In [ ]:
show_src(mdf._build_elrs_objects)

In [39]:
wifi_obj, rc_obj = mdf._build_elrs_objects(elrs)
if wifi_obj is not None:
    event.add_object(wifi_obj)
    show_attrs(wifi_obj)
    print()
else:
    print("No wifi-connection: SSID/password not present in this dump.\n")
if rc_obj is not None:
    event.add_object(rc_obj)
    show_attrs(rc_obj)

No wifi-connection: SSID/password not present in this dump.

<MISPObject name='remote-controller' uuid=aa8175cd attrs=3 refs=0>
  model                  = ExpressLRS receiver
  serial-number          = 55-49-9B-D7-0C-36
  remote-controller-id   = 55-49-9B-D7-0C-36


## 6. Reference graph

All objects hang off `uav`. `_add_references` applies the canonical edge set (skipping edges whose endpoints don't exist):

```
uav              --contains-->      iot-device
uav              --navigates-->     gpx
iot-firmware     --related-to-->    iot-device
file (FC)        --derived-from-->  uav
file (FC)        --related-to-->    iot-firmware
file (blackbox)  --derived-from-->  uav
geolocation(s)   --related-to-->    gpx
wifi-connection  --related-to-->    remote-controller
remote-controller --controls-->     uav
file (ELRS)      --derived-from-->  wifi-connection  (or uav, if no wifi)
```

In [ ]:
show_src(mdf._add_references)

In [ ]:
mdf._add_references(
    uav=uav,
    fc_file=fc_file, iot_device=iot_device, iot_firmware=iot_firmware, fc_gpx=fc_gpx,
    blackbox_file=blackbox_file, blackbox_gpx=blackbox_gpx,
    geolocation_objs=geolocation_objs,
    elrs_file=elrs_file, wifi_obj=wifi_obj, rc_obj=rc_obj,
)

by_uuid = {o.uuid: o for o in event.objects}
for obj in event.objects:
    for r in obj.references:
        tgt = by_uuid.get(r.referenced_uuid)
        print(f"{obj.name:<18} --[{r.relationship_type}]--> {tgt.name if tgt else '?'}")

## 7. GPS trace on a map

Visualisation only — not part of the MISP event. INAV waypoints from the FC, plus arming / disarming points from the blackbox.

In [41]:
import folium

def _coord(c):
    if not c:
        return None
    try:
        return float(c['lat']), float(c['lon'])
    except (TypeError, ValueError, KeyError):
        return None

points = []
for wp in fc.get("waypoints", []):
    pair = _coord(wp)
    if pair:
        points.append((*pair, f"WP {wp.get('index')} — {wp.get('action','')}"))

for log in bb.get("logs", []):
    for key, label in (("arming_coord", "Arming"), ("disarmed_coord", "Disarming")):
        pair = _coord(log.get(key))
        if pair:
            points.append((*pair, label))

if points:
    m = folium.Map(location=[points[0][0], points[0][1]], zoom_start=16)
    for lat, lon, tip in points:
        folium.Marker([lat, lon], tooltip=tip).add_to(m)
    if len(points) >= 2:
        folium.PolyLine([(lat, lon) for lat, lon, _ in points],
                        color="red", weight=3, opacity=0.8).add_to(m)
    display(m)
else:
    print("No GPS points available for this case.")

## 8. Final event

In [42]:
print(f"{len(event.objects)} MISP objects in event {event.info!r}\n")
for obj in event.objects:
    print(f"  - {obj.name:<18} ({len(obj.attributes)} attrs, {len(obj.references)} refs)")

10 MISP objects in event 'Drone forensic — forensic-02'

  - file               (5 attrs, 2 refs)
  - iot-device         (3 attrs, 1 refs)
  - iot-firmware       (5 attrs, 1 refs)
  - uav                (5 attrs, 0 refs)
  - file               (5 attrs, 1 refs)
  - gpx                (13 attrs, 1 refs)
  - geolocation        (3 attrs, 1 refs)
  - geolocation        (3 attrs, 1 refs)
  - file               (5 attrs, 0 refs)
  - remote-controller  (3 attrs, 0 refs)


In [43]:
out_path = REPO / "workshop" / f"event-{CASE}.json"
out_path.write_text(event.to_json(indent=2), encoding="utf-8")
print("Wrote", out_path)

Wrote /Users/chrisr3d/git/CIRCL/clean-drone-forensic/workshop/event-forensic-02.json
